In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Seq2Seq结构

- 整体结构：

    网络结构主要由编码器和解码器两部分组成，整个应用场景是将一个样本/序列的输入转换为另外一个样本/序列的输出，比如：翻译、古诗生成、对联生成、阅读理解、关键词抽取、文本生成等；

- 编码器

    最原始的编码器一般由RNN/LSTM/GRU结构组成，输入\[bs,et]形状的原始输入token id列表，得到每个token对应的高阶特征向量以及每个文本对应的文本特征向量；bs表示一个批次中存在bs个样本，每个样本由et个token组成，__PS:每个样本实际token数目不一致，所以一个批次中可能存在填充数据；__

    案例：一个样本、5个token组成的输入数据```[[12,35,26,34,253]]```

- 解码器

    最原始的解码器一般由RNN/LSTM/GRU结构组成，输入\[bs,dt]形状的解码器token id列表，对应实际token id列表shape也是\[bs,dt]，并且解码器的输入和解码器的输出恰好有一个位置的错位，并且解码器的第一个时刻的输入和最后一个时刻的输出一般都是特殊token id；bs表示一个批次中存在bs个样本，每个样本由dt个token组成，__PS:每个样本实际token数目不一致，所以一个批次中可能存在填充数据；__

    __NOTE:解码器必须为单向结构__ 

    案例：一个样本，6个token组成的解码器数据：

        解码器输入：[[3,102,235,1523,2132,1243]]
        解码器输出：[[102,235,1523,2132,1243,4]]

In [4]:
class EncoderModule(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers):
        super().__init__()
        # 词嵌入层：将离散的词汇 ID 映射为稠密的特征向量
        self.embed_layer = nn.Embedding(
            num_embeddings=vocab_size, # 词汇表大小 --> token到id转换的映射表大小
            embedding_dim=hidden_size # 每个词/Token对应的特征向量维度大小
        )
        # RNN 层：这里使用的是双向 LSTM (长短期记忆网络)
        self.rnn_layer = nn.LSTM(
            input_size = hidden_size, # 每个token输入的特征向量维度大小
            hidden_size = hidden_size,  # 每个token输出的特征向量维度大小
            num_layers = num_layers, # 层数
            batch_first = True, 
            bidirectional = True # 开启双向 LSTM（从左到右，以及从右到左）
        )
        # 特征转换层：将 RNN 提取的状态特征映射为最终的高阶语义向量
        # 将RNN输出特征值转换为高阶特征向量C
        self.ctx_feature_layer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU()
        )

    def forward(self, token_ids):
        # 1. token id转换为token embedding向量 [bs,et] -> [bs,et,hidden_size]
        token_embed = self.embed_layer(token_ids)
        # 2. 调用rnn结构获取序列特征向量
        # output: [bs,et,2*hidden_size] --> 当前是双向结构
        # state: rnn和gru的时候，只有一个值；lstm的时候，有两个值(二元组)；shape均为[?,bs,hidden_size]
        output, state = self.rnn_layer(token_embed)
        # 处理 LSTM 的最终状态 state
        if isinstance(state, tuple):
            # 将 LSTM 的隐状态 h_n 和 细胞状态 c_n 简单相加（一种特征融合方式）
            state = state[0] + state[1]
        # 此时 state 形状为 [层数*方向数, batch_size, hidden_size]
        # 沿着第 0 维度（层数*方向数）求平均，将多层多方向的状态压缩成一个单一的向量表示
        state = torch.mean(state, dim=0) # [?,bs,hidden_size] -->  [bs,hidden_size]
        # 3. 将状态信息转换为文本特征向量 [bs,hidden_size] -> [bs,hidden_size]
        ctx_embed = self.ctx_feature_layer(state)
        
        return ctx_embed

In [5]:
# 编码器案例
encoder = EncoderModule(100, 128, 3)
token_ids = torch.randint(0, 20, size=(2,5))
encoder_ctx_embed = encoder(token_ids)
print(encoder_ctx_embed.shape)

torch.Size([2, 128])


In [6]:
class DecoderModule(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers):
        super().__init__()
        self.num_layers = num_layers

        # 词嵌入层：将解码器的输入词 id 转为词向量
        self.embed_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=hidden_size)

        # 状态初始化层：将 Encoder 的上下文向量映射为 Decoder LSTM 的初始隐状态 h0 和细胞状态 c0
        # 由于 LSTM 需要多层的初始状态，所以输出维度扩大为 num_layers * hidden_size
        self.rnn_init_h0_layers = nn.Linear(hidden_size, num_layers * hidden_size)
        self.rnn_init_c0_layers = nn.Linear(hidden_size, num_layers * hidden_size)
        
        # 解码 LSTM 层：注意解码器必须是单向的 (bidirectional = False)
        self.rnn_layer = nn.LSTM(
            input_size = hidden_size, hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True, bidirectional = False 
        )
        
        # 分类层：将 LSTM 的输出映射回词汇表大小的向量，用于预测下一个词的概率分布
        self.classify_layer = nn.Linear(hidden_size, vocab_size)

    def forward(self, token_ids, encoder_ctx):
        """
            前向执行过程
            : token_ids : [batch_size, seq_len] 解码器输入的 token ids (训练时包含完整目标序列，推理时通常是起始符 <BOS>)
            : encoder_ctx : [batch_size, hidden_size] 编码器提取出的文本特征向量
        """
        # 1. 将编码器的上下文向量转换为解码器 LSTM 需要的初始状态格式 (h0, c0)
        bs, e = encoder_ctx.shape # e 即 hidden_size
        
        # 处理 h0 (隐状态)
        h0 = self.rnn_init_h0_layers(encoder_ctx) # [bs, hidden_size] -> [bs, num_layers * hidden_size]
        h0 = h0.reshape((bs, e, -1))              # 拆分为 [bs, hidden_size, num_layers]
        h0 = torch.permute(h0, dims=(2, 0, 1))    # 调整维度以满足 LSTM 要求: [num_layers, bs, hidden_size]

        # 处理 c0 (细胞状态)
        c0 = self.rnn_init_c0_layers(encoder_ctx) # [bs, hidden_size] -> [bs, num_layers * hidden_size]
        c0 = c0.reshape((bs, e, -1))              # 拆分为 [bs, hidden_size, num_layers]
        c0 = torch.permute(c0, dims=(2, 0, 1))    # 调整维度以满足 LSTM 要求: [num_layers, bs, hidden_size]

        if self.training:
            # ---------------- 训练模式 (Teacher Forcing 模式) ----------------
            # 训练时，一次性将所有时间步的真实序列输入给模型
            # 2. 将输入序列转为词向量: [bs, seq_len] -> [bs, seq_len, hidden_size]
            token_embed = self.embed_layer(token_ids)
    
            # 3. 输入 LSTM 获取输出特征
            # output: [bs, seq_len, hidden_size] 包含每个时间步的特征
            output, _ = self.rnn_layer(token_embed, (h0, c0)) # 传入 h0, c0 作为初始状态
    
            # 4. 通过全连接层计算每个时间步预测下一个词的得分
            # [bs, seq_len, hidden_size] -> [bs, seq_len, vocab_size]
            score = self.classify_layer(output)
            
            return score # 返回预测得分，后续用于计算交叉熵损失
            
        else:
            # ---------------- 推理/预测模式 (自回归生成模式) ----------------
            # 推理时，无法提前知道未来的词，只能基于当前已有的词一步步向后预测
            while True:
                # 2. 将当前已生成的序列转为词向量: [bs, seq_len] -> [bs, seq_len, hidden_size]
                token_embed = self.embed_layer(token_ids)

                # 3. 输入 LSTM。注意：在简单的循环中，每次都重新输入了从头到尾的完整生成序列
                # 并在每一步使用最初始的 (h0, c0)。(更高效的做法是只输入上一步的输出并传递内部状态，但这为了逻辑清晰简化了代码)
                output, _ = self.rnn_layer(token_embed, (h0, c0))

                # 4. 我们只关心 LSTM 在当前序列的最后一个时刻的输出特征
                output_t = output[:, -1, :] # 切片提取最后一个时间步: [bs, hidden_size]

                # 5. 预测最后一个时间步下一个词的各类得分
                score = self.classify_layer(output_t) # [bs, vocab_size]

                # 6. 获取得分最高的类别索引作为预测的词 ID
                pred_ids = torch.argmax(score, dim=1, keepdim=True) # [bs, 1] 

                # 7. 将新预测出来的词 ID 拼接在现有序列的末尾，作为下一轮循环的输入
                token_ids = torch.cat([token_ids, pred_ids], dim=1) # [bs, seq_len + 1]

                # 8. 终止条件判断：如果生成的序列长度超过 10，则强行停止生成。
                # (实际业务中还会判断 pred_ids 是否为特定的结束符如 <EOS> 或 <SEP>)
                if token_ids.shape[1] > 10:
                    break
                    
            return token_ids # 返回最终生成的包含多个 token ID 的完整序列

In [7]:
# 解码器案例
decoder = DecoderModule(100, 128, 3)
token_ids = torch.randint(0, 20, size=(2,5)) # 解码器的输入
encoder_ctx_embed = torch.rand(2,128) # 编码器提取出来的特征向量

decoder_score = decoder(token_ids, encoder_ctx_embed)
print(f"解码器输出:{decoder_score.shape}")

解码器输出:torch.Size([2, 5, 100])


In [8]:
class Seq2SeqModule(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, decoder_vocab_size=None, decoder_num_layers=None):
        super().__init__()
        
        # 允许解码器有不同的词汇表大小和层数（例如中英翻译，两边词表大小不同）
        if decoder_vocab_size is None:
            decoder_vocab_size = vocab_size
        if decoder_num_layers is None:
            decoder_num_layers = num_layers
            
        # 实例化编码器
        self.encoder = EncoderModule(vocab_size, hidden_size, num_layers)
        # 实例化解码器
        self.decoder = DecoderModule(decoder_vocab_size, hidden_size, decoder_num_layers)

    def forward(self, encoder_token_ids, decoder_token_ids):
        # 1. 编码器前向传播，提取源句子的上下文特征向量
        encoder_ctx = self.encoder(encoder_token_ids)

        # 2. 解码器前向传播，接收上下文向量并结合当前已知序列生成目标结果
        decoder_outputs = self.decoder(decoder_token_ids, encoder_ctx)

        return decoder_outputs # 训练时返回 score, 推理时返回 token 序列

In [11]:
# Seq2Seq案例
# 初始化整个 Seq2Seq 模型
seq2seq = Seq2SeqModule(10000, 64, 3, decoder_num_layers=2)
print(seq2seq)
# 定义交叉熵损失函数，reduction='none' 表示不求平均，保留每个 token 的损失方便查看
loss_fn = nn.CrossEntropyLoss(reduction='none')
#loss_fn = nn.CrossEntropyLoss()

Seq2SeqModule(
  (encoder): EncoderModule(
    (embed_layer): Embedding(10000, 64)
    (rnn_layer): LSTM(64, 64, num_layers=3, batch_first=True, bidirectional=True)
    (ctx_feature_layer): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (decoder): DecoderModule(
    (embed_layer): Embedding(10000, 64)
    (rnn_init_h0_layers): Linear(in_features=64, out_features=128, bias=True)
    (rnn_init_c0_layers): Linear(in_features=64, out_features=128, bias=True)
    (rnn_layer): LSTM(64, 64, num_layers=2, batch_first=True)
    (classify_layer): Linear(in_features=64, out_features=10000, bias=True)
  )
)


In [13]:
# 训练过程测试
# 假设编码器输入了 5 个 token (例如中文句子 "我 爱 中 国 人")
encoder_token_ids = torch.tensor([[12,35,26,34,253]]) # [batch_size=1, seq_len=5]

# 解码器输入，通常以特殊的起始符 (如 id=3，代表 <BOS>) 开头
# 形如: <BOS> I love Chi nese
decoder_token_ids = torch.tensor([[3,102,235,1523,2132,1243]]) # [1, 6]

# 解码器的真实目标序列，对应前一个序列往后错位一个 token，并以结束符 (如 id=4，代表 <EOS>) 结尾
# 形如: I love Chi nese <EOS>
decoder_target_ids = torch.tensor([[102,235,1523,2132,1243,4]]) # [1, 6]

seq2seq.train() # 开启训练模式
decoder_score = seq2seq(encoder_token_ids, decoder_token_ids) # 返回预测 logits, 形状: [1, 6, 10000]

# 计算损失。注意：CrossEntropyLoss 期望的预测值张量形状为 [batch_size, num_classes, ...]
# 所以这里使用 torch.permute 将类别维度放在第 1 维: [1, 6, 10000] -> [1, 10000, 6]
loss = loss_fn(torch.permute(decoder_score, dims=(0,2,1)), decoder_target_ids)
print(loss)

torch.Size([1, 6, 10000])
tensor([[9.2611, 9.2227, 9.3771, 9.3195, 9.2434, 9.1634]],
       grad_fn=<ViewBackward0>)


In [14]:
# 推理预测过程测试
# 同样输入一段源文本
encoder_token_ids = torch.tensor([[12,35,26,34,253]])

# 推理时，解码器的初始输入只有起始符 <BOS> (id=3)
decoder_token_ids = torch.tensor([[3]])

seq2seq.eval() # 切换至评估（推理）模式

# 模型会进入 while True 自回归循环，直到生成长度 > 10 停止
pred_token_ids = seq2seq(encoder_token_ids, decoder_token_ids)

# 打印最终生成的预测序列 (长度通常为 11)
# print(pred_token_ids.shape)

torch.Size([1, 11])
预测token id:
	tensor([[   3,  595, 2027, 2027, 4310, 2215,  595, 2215, 6286, 9048, 9048]])
